# Step 0: Install Required Libraries and Preprocessing

# Import necessary libraries and perform initial setup for data preprocessing.

In [ ]:
# pip install nibabel
import os
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# A function to read CT scan files and extract middle slices:

# Load and preprocess NIfTI CT scan files. Extract the middle slices and normalize them. The labels CT-3 and CT-4 are merged into a single class (CT-3) due to the small sample size of CT-4.

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm import tqdm # Add progress bar library

def load_and_preprocess_nifti(file_path, target_size=(128, 128)):
    # Load 3D file
    scan = nib.load(file_path).get_fdata()
    
    # ⚡ Important change: First we separate the middle slice
    mid_slice = scan[:, :, scan.shape[2] // 2]
    
    # ⚡ Now we normalize only that single 2D slice (much faster)
    mid_slice = (mid_slice - np.min(mid_slice)) / (np.max(mid_slice) - np.min(mid_slice) + 1e-8)
    
    # Change dimensions
    mid_slice = np.expand_dims(mid_slice, axis=-1)
    return tf.image.resize(mid_slice, target_size).numpy()

dataset_path = '/kaggle/input/datasets/mathurinache/mosmeddata-chest-ct-scans-with-covid19/MosMedData Chest CT Scans with COVID-19 Related Findings COVID19_1110 1.0/studies'
file_paths = glob.glob(os.path.join(dataset_path, '**', '*.nii*'), recursive=True)

X_data = []
y_data = []

print(f"Total files found: {len(file_paths)}")
print("Processing images... (with optimized algorithm)")

# Add tqdm to the loop to show a nice progress bar
for path in tqdm(file_paths, desc="Processing CT Scans", unit="file"):
    try:
        # Extract label from folder name (CT-0 to CT-4)
        folder_name = os.path.basename(os.path.dirname(path))
        if "CT-0" in folder_name: label = 0
        elif "CT-1" in folder_name: label = 1
        elif "CT-2" in folder_name: label = 2
        elif "CT-3" in folder_name: label = 3
        elif "CT-4" in folder_name: label = 3  # Merged CT-4 with CT-3 due to small sample size
        else: continue 
            
        img = load_and_preprocess_nifti(path)
        X_data.append(img)
        y_data.append(label)
    except Exception as e:
        # If a file is corrupted, we skip it
        pass

X = np.array(X_data)
y = np.array(y_data)

# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nProcessing completed successfully!")
print(f"Training data dimensions: X={X_train.shape}, y={y_train.shape}")
print(f"Test data dimensions: X={X_test.shape}, y={y_test.shape}")

# Define a data augmentation pipeline to artificially expand the training dataset and prevent overfitting. This includes mild rotation and zoom.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (accuracy_score, recall_score, f1_score, 
                             matthews_corrcoef, confusion_matrix, 
                             roc_auc_score, roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import label_binarize

# Data augmentation layer (runs very fast on CPU/GPU)
data_augmentation = tf.keras.Sequential([
    #layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05), # Very mild rotation (about 18 degrees)
    layers.RandomZoom(0.1)       # Mild 10% zoom
], name="data_augmentation")

# Step 1: Design and Train Autoencoder

# Define the architecture for the Convolutional Autoencoder. The encoder compresses the image into a 256-dimensional latent space, and the decoder reconstructs the image from these features.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# 1. Define Data Augmentation
data_augmentation = tf.keras.Sequential([
    #layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1)
], name="data_augmentation")

input_shape = (128, 128, 1)

# 2. Encoder Section
encoder_input = layers.Input(shape=input_shape, name="encoder_input")
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_input)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)

# Flattening layer and the most compressed part (Bottleneck / Features)
shape_before_flatten = tf.keras.backend.int_shape(x)[1:]
x = layers.Flatten()(x)
latent_space = layers.Dense(256, activation='relu', name="latent_features")(x) # 256 extracted features

encoder = models.Model(encoder_input, latent_space, name="encoder")

# 3. Decoder Section
decoder_input = layers.Input(shape=(256,), name="decoder_input")
x = layers.Dense(int(np.prod(shape_before_flatten)), activation='relu')(decoder_input)
x = layers.Reshape(shape_before_flatten)(x)

x = layers.Conv2DTranspose(128, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2DTranspose(64, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2DTranspose(32, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)
decoder_output = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

decoder = models.Model(decoder_input, decoder_output, name="decoder")

# 4. Final Autoencoder Model (Connecting components together)
autoencoder_input = layers.Input(shape=input_shape, name="autoencoder_input")

# Apply augmentation only during training (this layer is automatically disabled during inference)
augmented_input = data_augmentation(autoencoder_input) 

# Pass the image through the encoder and then decoder
encoded_img = encoder(augmented_input)
reconstructed_img = decoder(encoded_img)

autoencoder = models.Model(autoencoder_input, reconstructed_img, name="autoencoder")
autoencoder.compile(optimizer='adam', loss='mse')

# Print model summary
autoencoder.summary()

# Configure Early Stopping and Model Checkpoint callbacks, then train the Autoencoder model. After training, plot the loss and reconstruction error distribution, and extract the latent features.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- 1. Define Callbacks for Autoencoder ---
ae_early_stopping = EarlyStopping(
    monitor='val_loss',       # Monitor validation loss
    patience=7,               # If validation loss doesn't improve for 7 epochs, stop
    restore_best_weights=True # Finally, restore the best weights
)

ae_checkpoint = ModelCheckpoint(
    filepath='best_autoencoder.keras', # Filename to save the model
    monitor='val_loss', 
    save_best_only=True,               # Save only the model that performs better
    verbose=1
)

# --- 2. Train Autoencoder with Callbacks ---
print("Start autoencoder training...")
history_ae = autoencoder.fit(
    X_train, X_train, 
    epochs=50, 
    batch_size=32, 
    validation_data=(X_test, X_test),
    callbacks=[ae_early_stopping, ae_checkpoint] # <<< Apply functions here
)

'''
# Train autoencoder
history_ae = autoencoder.fit(
    X_train, X_train, 
    epochs=300, 
    batch_size=16, 
    validation_data=(X_test, X_test)
)
'''

# ---- Autoencoder Evaluation: Plot Loss and Reconstruction Error Distribution ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
axes[0].plot(history_ae.history['loss'], label='Train MSE')
axes[0].plot(history_ae.history['val_loss'], label='Validation MSE')
axes[0].set_title('Autoencoder Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Squared Error')
axes[0].legend()

# Histogram of reconstruction error on test data
reconstructed_test = autoencoder.predict(X_test)
mse_errors = np.mean(np.square(X_test - reconstructed_test), axis=(1, 2, 3))
axes[1].hist(mse_errors, bins=50, color='blue', alpha=0.7)
axes[1].set_title('Reconstruction Error Distribution (Test Data)')
axes[1].set_xlabel('MSE')
axes[1].set_ylabel('Number of Samples')

plt.tight_layout()
plt.show()

# Extract features for next steps
X_train_features = encoder.predict(X_train)
X_test_features = encoder.predict(X_test)

# Visualize the performance of the Autoencoder by comparing original test images with their reconstructed counterparts.

In [ ]:
# Predict (reconstruct) test images
reconstructed_imgs = autoencoder.predict(X_test)

# Plot 5 random images: first row original, second row reconstructed
n = 5
plt.figure(figsize=(15, 6))
for i in range(n):
    # Original image
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(X_test[i].squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis("off")
    
    # Reconstructed image
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed_imgs[i].squeeze(), cmap='gray')
    plt.title("Reconstructed")
    plt.axis("off")
plt.tight_layout()
plt.show()

# Step 2: Extract Features using Encoder

# Apply Principal Component Analysis (PCA) to reduce the 256-dimensional latent features into 2 dimensions, and plot them to observe the clustering of different classes.

In [ ]:
from sklearn.decomposition import PCA

# Extract 256 features
X_train_features = encoder.predict(X_train)
X_test_features = encoder.predict(X_test)

print(f"Shape of extracted features: {X_train_features.shape}")

# Reduce dimensionality from 256 to 2 to display on a 2D plot
pca = PCA(n_components=2)
features_2d = pca.fit_transform(X_test_features)

# Plot features (same color means similar disease class)
plt.figure(figsize=(10, 8))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=y_test, cmap='jet', alpha=0.7)
plt.colorbar(scatter, label='COVID-19 Severity (0 to 4)')
plt.title('Visualization of Extracted Features (PCA applied on 256-D space)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

# Address class imbalance in the training data by using Random Over Sampler. This ensures all classes have an equal number of samples during classifier training.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
import numpy as np

# Define data oversampler
ros = RandomOverSampler(random_state=42)

# Balance training data 
X_train_features_bal, y_train_bal = ros.fit_resample(X_train_features, y_train)

print("📊 Training data distribution before balancing:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts): print(f"Class {u}: {c} samples")

print("-" * 40)
print("⚖️ Training data distribution after balancing:")
unique_bal, counts_bal = np.unique(y_train_bal, return_counts=True)
for u, c in zip(unique_bal, counts_bal): print(f"Class {u}: {c} samples")

# Define a comprehensive evaluation function `evaluate_classifier_comprehensive` that calculates various metrics (Accuracy, Precision, Recall, F1-Score, ROC AUC, etc.) and plots Confusion Matrix, ROC curves, and Precision-Recall curves.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             matthews_corrcoef, confusion_matrix, roc_auc_score, 
                             roc_curve, auc, precision_recall_curve, average_precision_score,
                             classification_report)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

mosmed_classes = ['CT-0 (Normal)', 'CT-1 (Mild)', 'CT-2 (Moderate)', 'CT-3 (Severe/Critical)'] # Merged CT-3 and CT-4
def evaluate_classifier_comprehensive(y_true, y_pred, y_prob, class_names):
    n_classes = len(class_names)
    labels = range(n_classes) # Force checking all 4 classes
    
    y_true_bin = label_binarize(y_true, classes=labels)
    
    print("=" * 60)
    print(" 📊 Comprehensive Model Evaluation Report (Classification Report)")
    print("=" * 60)
    # Added labels parameter
    print(classification_report(y_true, y_pred, labels=labels, target_names=class_names, digits=4, zero_division=0))
    
    print("=" * 60)
    print(" 🎯 Macro and Micro metrics")
    print("=" * 60)
    
    print(f"Accuracy:                  {accuracy_score(y_true, y_pred):.4f}")
    print(f"MCC:                       {matthews_corrcoef(y_true, y_pred):.4f}")
    print("-" * 60)
    
    print(f"Precision (Macro):         {precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.4f}")
    print(f"Precision (Micro):         {precision_score(y_true, y_pred, labels=labels, average='micro', zero_division=0):.4f}")
    print(f"Recall (Macro):            {recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.4f}")
    print(f"Recall (Micro):            {recall_score(y_true, y_pred, labels=labels, average='micro', zero_division=0):.4f}")
    print(f"F1-Score (Macro):          {f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.4f}")
    print(f"F1-Score (Micro):          {f1_score(y_true, y_pred, labels=labels, average='micro', zero_division=0):.4f}")
    print("-" * 60)
    
    print(f"ROC AUC (Macro):           {roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr'):.4f}")
    print(f"ROC AUC (Micro):           {roc_auc_score(y_true_bin, y_prob, average='micro', multi_class='ovr'):.4f}")
    print(f"PR AUC (Macro):            {average_precision_score(y_true_bin, y_prob, average='macro'):.4f}")
    print(f"PR AUC (Micro):            {average_precision_score(y_true_bin, y_prob, average='micro'):.4f}")
    print("=" * 60)
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    
    # Added labels parameter to ensure matrix stays 4x4
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    axes[0].set_title('Confusion Matrix', fontsize=14)
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('True Label')
    axes[0].tick_params(axis='x', rotation=45)
    
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        axes[1].plot(fpr, tpr, lw=1.5, label=f'{class_names[i]} (AUC = {auc(fpr, tpr):.2f})')
        
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    axes[1].plot(fpr_micro, tpr_micro, lw=2.5, linestyle=':', color='black', 
                 label=f'Micro-average (AUC = {auc(fpr_micro, tpr_micro):.2f})')
    
    axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
    axes[1].set_title('ROC Curve (Multi-class)', fontsize=14)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].legend(loc="lower right", fontsize=9)
    axes[1].grid(alpha=0.3)
    
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_prob[:, i])
        axes[2].plot(recall, precision, lw=1.5, label=f'{class_names[i]}')
        
    precision_micro, recall_micro, _ = precision_recall_curve(y_true_bin.ravel(), y_prob.ravel())
    pr_micro_auc = average_precision_score(y_true_bin, y_prob, average="micro")
    axes[2].plot(recall_micro, precision_micro, lw=2.5, linestyle=':', color='black', 
                 label=f'Micro-average (AUC = {pr_micro_auc:.2f})')
                 
    axes[2].set_title('Precision-Recall Curve', fontsize=14)
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].legend(loc="lower left", fontsize=9)
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Design and train a Neural Network Classifier using the balanced latent features. Callbacks are used to prevent overfitting and save the best model. Finally, the model is evaluated.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt

# Design neural network classifier
classifier_nn = models.Sequential([
    layers.Input(shape=(256,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4), # Increase Dropout to prevent Overfit
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax') # 4 classes (CT-0 to CT-3)
])

classifier_nn.compile(optimizer='adam', 
                      loss='sparse_categorical_crossentropy', 
                      metrics=['accuracy'])

# --- 1. Define Callbacks for Classifier Network ---
clf_early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=10, # In classification we usually wait a bit longer (e.g. 10)
    restore_best_weights=True
)

clf_checkpoint = ModelCheckpoint(
    filepath='best_classifier_nn.keras', # Classifier save filename
    monitor='val_loss', 
    save_best_only=True,
    verbose=1
)

# --- 2. Train Neural Network Classifier with Callbacks ---
print("Start training classifier network...")
history_clf_bal = classifier_nn.fit(
    X_train_features_bal, y_train_bal, 
    epochs=100, 
    batch_size=8, 
    validation_data=(X_test_features, y_test),
    callbacks=[clf_early_stopping, clf_checkpoint], 
    verbose=1
)

# --- 3. Visualize Classifier Training Progress ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Plot
axes[0].plot(history_clf_bal.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_clf_bal.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Classifier Accuracy')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss Plot
axes[1].plot(history_clf_bal.history['loss'], label='Train Loss')
axes[1].plot(history_clf_bal.history['val_loss'], label='Validation Loss')
axes[1].set_title('Classifier Loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.show()

# --- 4. Comprehensive Evaluation with Custom Function ---
# A) First generate model predictions on test data
y_prob_nn = classifier_nn.predict(X_test_features)
y_pred_nn = np.argmax(y_prob_nn, axis=1)

# B) Call evaluation function with class names
mosmed_classes = ['CT-0 (Normal)', 'CT-1 (Mild)', 'CT-2 (Moderate)', 'CT-3 (Severe/Critical)'] # Merged CT-3 and CT-4

print("\n>>> 🧠 Comprehensive Evaluation of Neural Network Classifier <<<")
evaluate_classifier_comprehensive(y_test, y_pred_nn, y_prob_nn, class_names=mosmed_classes)

# Step 3: Train Classification Model

> Method 1: Using Random Forest 

# Train a Random Forest Classifier as a baseline model using the balanced latent features and evaluate its performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Build and train model
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42)
rf_classifier.fit(X_train_features_bal, y_train_bal)

# Predict class and probabilities
y_pred_rf = rf_classifier.predict(X_test_features)
y_prob_rf = rf_classifier.predict_proba(X_test_features)

print(">>> Random Forest Model Evaluation <<<")
evaluate_classifier_comprehensive(y_test, y_pred_rf, y_prob_rf, class_names=mosmed_classes)

> Method 2: Using a Small Neural Network (Deep Learning)

# Implement a Learning Rate Finder to find the optimal learning rate for the Neural Network classifier by gradually increasing the learning rate and observing the loss.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import LearningRateScheduler
import matplotlib.pyplot as plt

# 1. Define a temporary model exactly like your main model
lr_model = models.Sequential([
    layers.Input(shape=(256,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')
])

# 2. Function that exponentially increases learning rate every Epoch
# Start from 1e-5 and reach larger values
lr_schedule = LearningRateScheduler(lambda epoch: 1e-5 * 10**(epoch / 20))

# Compile with base learning rate
lr_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), 
                 loss='sparse_categorical_crossentropy', 
                 metrics=['accuracy'])

print("🔍 Searching for the best learning rate... (this step takes some time)")
# 3. Temporary training (only 100 epochs)
history_lr = lr_model.fit(
    X_train_features_bal, y_train_bal, 
    epochs=100, 
    batch_size=16, 
    callbacks=[lr_schedule],
    verbose=0 
)

# 4. Plot chart
plt.figure(figsize=(10, 6))
# Use logarithmic scale for X-axis to better see changes
plt.semilogx(history_lr.history['learning_rate'], history_lr.history['loss'], lw=2, color='blue')
plt.title('Learning Rate Finder', fontsize=14)
plt.xlabel('Learning Rate (Log Scale)')
plt.ylabel('Loss (Error)')
plt.grid(True, which="both", ls="--", alpha=0.6)

# Limit Y-axis to prevent chart cluttering when error explodes
plt.ylim([min(history_lr.history['loss']) - 0.2, history_lr.history['loss'][0] + 0.5])
plt.show()

# Train the final Neural Network classifier using the optimal learning rate found in the previous step. Evaluate its performance comprehensively.

In [ ]:
from tensorflow.keras.optimizers import Adam

best_lr = 0.001 # <<< Change this number based on your own chart


classifier_nn = models.Sequential([
    layers.Input(shape=(256,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')
])

classifier_nn.compile(optimizer=Adam(learning_rate=best_lr), 
                      loss='sparse_categorical_crossentropy', 
                      metrics=['accuracy'])


# --- 1. Define Callbacks ---
clf_early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=15,               # If validation loss doesn't decrease for 15 steps, stop
    restore_best_weights=True  # Restore best weights at the end
)

clf_checkpoint = ModelCheckpoint(
    filepath='best_classifier_nn.keras', 
    monitor='val_loss', 
    save_best_only=True,       # Save only the best model
    verbose=1
)

# --- 2. Train Model and Apply Callbacks ---
print("Start training classifier network...")
history_clf = classifier_nn.fit(
    X_train_features_bal, y_train_bal, 
    epochs=300, 
    batch_size=16, 
    validation_data=(X_test_features, y_test),
    callbacks=[clf_early_stopping, clf_checkpoint], # <<< Apply here
    verbose=1 
)


# Plot training charts (Train vs Validation)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_clf.history['accuracy'], label='Train Accuracy')
axes[0].plot(history_clf.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('NN Classifier - Accuracy')
axes[0].legend()

axes[1].plot(history_clf.history['loss'], label='Train Loss')
axes[1].plot(history_clf.history['val_loss'], label='Validation Loss')
axes[1].set_title('NN Classifier - Loss')
axes[1].legend()
plt.show()

# Predict class and probabilities for comprehensive evaluation
y_prob_nn = classifier_nn.predict(X_test_features)
y_pred_nn = np.argmax(y_prob_nn, axis=1)

print(">>> Neural Network Model Evaluation <<<")
evaluate_classifier_comprehensive(y_test, y_pred_nn, y_prob_nn, class_names=mosmed_classes)

# Empty code cell.